# Multispectral Data Preprocessing for CVAT Labeling

This notebook processes multispectral imagery and vegetation indices to prepare data for:
1. CVAT labeling (RGB images exported as PNG)
2. Deep learning training (multispectral bands + indices stored alongside)

## Workflow:
- Load HD_map (5-band RGB + NIR + RedEdge) and index_map (NDVI, GNDVI, NDRE, LCI, OSAVI)
- Tile all data into 256x256 patches
- Export RGB tiles as PNG for CVAT labeling
- Save multispectral bands and indices as .npy files for training
- Filter out low-quality tiles (too dark, homogeneous)

In [ ]:
import os
import numpy as np
import rasterio
from rasterio.windows import Window
import cv2
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
from typing import Tuple, Dict, List
import warnings
warnings.filterwarnings('ignore')

## Configuration

In [ ]:
# Paths
FLIGHT_NAME = 'flight_02'
BASE_DIR = Path('data/processed_multispectral') / FLIGHT_NAME
HD_MAP_DIR = BASE_DIR / 'HD_map'
INDEX_MAP_DIR = BASE_DIR / 'index_map'
OUTPUT_DIR = Path('data/cvat_ready_flight_02') / FLIGHT_NAME

# Tiling parameters
TILE_SIZE = 256
OVERLAP = 0  # Can be increased for better edge detection

# Quality filtering thresholds
MIN_BRIGHTNESS = 20  # Minimum mean brightness (0-255)
MIN_STD = 5  # Minimum standard deviation (avoid homogeneous areas)

# Create output directories
RGB_OUTPUT_DIR = OUTPUT_DIR / 'rgb_tiles'
BANDS_OUTPUT_DIR = OUTPUT_DIR / 'band_tiles'
INDEX_OUTPUT_DIR = OUTPUT_DIR / 'index_tiles'
METADATA_OUTPUT_DIR = OUTPUT_DIR / 'metadata'

for dir_path in [RGB_OUTPUT_DIR, BANDS_OUTPUT_DIR, INDEX_OUTPUT_DIR, METADATA_OUTPUT_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print(f"HD Map Directory: {HD_MAP_DIR}")
print(f"Index Map Directory: {INDEX_MAP_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Tile Size: {TILE_SIZE}x{TILE_SIZE}")

## Data Exploration

In [ ]:
# Load and inspect the main RGB image
rgb_path = HD_MAP_DIR / 'result.tif'

with rasterio.open(rgb_path) as src:
    print("RGB Image (result.tif):")
    print(f"  Shape: {src.height} x {src.width}")
    print(f"  Bands: {src.count}")
    print(f"  CRS: {src.crs}")
    print(f"  Bounds: {src.bounds}")
    print(f"  Data type: {src.dtypes[0]}")
    
    # Calculate number of tiles
    stride = TILE_SIZE - OVERLAP
    n_tiles_h = (src.height - TILE_SIZE) // stride + 1
    n_tiles_w = (src.width - TILE_SIZE) // stride + 1
    total_tiles = n_tiles_h * n_tiles_w
    print(f"\nExpected tiles: {n_tiles_h} x {n_tiles_w} = {total_tiles}")

In [ ]:
# List available individual bands
print("\nAvailable individual bands in HD_map:")
band_files = sorted(HD_MAP_DIR.glob('result_*.tif'))
for bf in band_files:
    with rasterio.open(bf) as src:
        print(f"  {bf.name}: {src.height} x {src.width}, dtype: {src.dtypes[0]}")

In [ ]:
# List available vegetation indices
print("\nAvailable vegetation indices:")
index_files = sorted(INDEX_MAP_DIR.glob('*.tif'))
index_files = [f for f in index_files if not f.name.endswith('.aux.xml')]

for idx_file in index_files:
    with rasterio.open(idx_file) as src:
        print(f"  {idx_file.stem}: {src.height} x {src.width}, dtype: {src.dtypes[0]}")

## Utility Functions

In [ ]:
def normalize_to_uint8(array: np.ndarray, percentile_clip: Tuple[float, float] = (2, 98)) -> np.ndarray:
    """
    Normalize array to 0-255 uint8 range using percentile clipping.
    """
    if array.size == 0:
        return array
    
    # Handle NaN and inf values
    valid_mask = np.isfinite(array)
    if not valid_mask.any():
        return np.zeros_like(array, dtype=np.uint8)
    
    valid_data = array[valid_mask]
    
    # Percentile clipping for better contrast
    p_low, p_high = np.percentile(valid_data, percentile_clip)
    
    # Clip and normalize
    clipped = np.clip(array, p_low, p_high)
    
    if p_high - p_low > 0:
        normalized = ((clipped - p_low) / (p_high - p_low) * 255).astype(np.uint8)
    else:
        normalized = np.zeros_like(array, dtype=np.uint8)
    
    return normalized


def is_valid_tile(tile: np.ndarray, min_brightness: float = MIN_BRIGHTNESS, 
                  min_std: float = MIN_STD) -> bool:
    """
    Check if a tile meets quality criteria.
    """
    if tile.size == 0:
        return False
    
    # Check brightness
    mean_brightness = np.mean(tile)
    if mean_brightness < min_brightness:
        return False
    
    # Check variance (avoid uniform areas)
    std_dev = np.std(tile)
    if std_dev < min_std:
        return False
    
    return True


def extract_tiles_from_raster(raster_path: Path, tile_size: int = TILE_SIZE, 
                              overlap: int = OVERLAP) -> List[Tuple[np.ndarray, int, int]]:
    """
    Extract tiles from a raster file.
    Returns list of (tile_data, row_idx, col_idx)
    """
    tiles = []
    stride = tile_size - overlap
    
    with rasterio.open(raster_path) as src:
        height, width = src.height, src.width
        
        for row in range(0, height - tile_size + 1, stride):
            for col in range(0, width - tile_size + 1, stride):
                window = Window(col, row, tile_size, tile_size)
                tile_data = src.read(window=window)
                
                # Handle different band counts
                if tile_data.shape[0] == 1:
                    tile_data = tile_data[0]  # Single band
                else:
                    tile_data = np.transpose(tile_data, (1, 2, 0))  # Multi-band (H, W, C)
                
                tiles.append((tile_data, row // stride, col // stride))
    
    return tiles

## Process RGB Tiles for CVAT

In [ ]:
print("Processing RGB tiles for CVAT labeling...")

rgb_tiles = extract_tiles_from_raster(rgb_path, TILE_SIZE, OVERLAP)
valid_tile_indices = []
tile_metadata = []

for tile_data, row_idx, col_idx in tqdm(rgb_tiles, desc="Processing RGB tiles"):
    # Normalize to uint8 for visualization
    if tile_data.shape[-1] >= 3:
        # Extract RGB channels (assuming order is R, G, B or similar)
        rgb_tile = tile_data[:, :, :3].copy()
    else:
        print(f"Warning: Tile at ({row_idx}, {col_idx}) has fewer than 3 bands")
        continue
    
    # Normalize each channel
    rgb_normalized = np.stack([
        normalize_to_uint8(rgb_tile[:, :, 0]),
        normalize_to_uint8(rgb_tile[:, :, 1]),
        normalize_to_uint8(rgb_tile[:, :, 2])
    ], axis=-1)
    
    # Quality check
    if not is_valid_tile(rgb_normalized):
        continue
    
    # Save RGB tile for CVAT
    tile_name = f"tile_{row_idx:04d}_{col_idx:04d}"
    rgb_output_path = RGB_OUTPUT_DIR / f"{tile_name}.png"
    
    # Convert RGB to BGR for OpenCV
    bgr_tile = cv2.cvtColor(rgb_normalized, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(rgb_output_path), bgr_tile)
    
    valid_tile_indices.append((row_idx, col_idx))
    tile_metadata.append({
        'tile_name': tile_name,
        'row_idx': row_idx,
        'col_idx': col_idx,
        'mean_brightness': float(np.mean(rgb_normalized)),
        'std_dev': float(np.std(rgb_normalized))
    })

print(f"\nValid tiles saved: {len(valid_tile_indices)} / {len(rgb_tiles)}")
print(f"Rejection rate: {(1 - len(valid_tile_indices)/len(rgb_tiles))*100:.1f}%")

## Process Multispectral Bands

In [ ]:
print("\nProcessing individual multispectral bands...")

# Define band files to process
band_mapping = {
    'green': HD_MAP_DIR / 'result_Green.tif',
    'red': HD_MAP_DIR / 'result_Red.tif',
    'rededge': HD_MAP_DIR / 'result_RedEdge.tif',
    'nir': HD_MAP_DIR / 'result_NIR.tif'
}

# Process each band
for band_name, band_path in band_mapping.items():
    if not band_path.exists():
        print(f"Warning: {band_path} not found, skipping...")
        continue
    
    print(f"Processing {band_name} band...")
    band_tiles = extract_tiles_from_raster(band_path, TILE_SIZE, OVERLAP)
    
    for tile_data, row_idx, col_idx in tqdm(band_tiles, desc=f"Tiling {band_name}"):
        # Only save tiles that passed RGB quality check
        if (row_idx, col_idx) not in valid_tile_indices:
            continue
        
        tile_name = f"tile_{row_idx:04d}_{col_idx:04d}"
        band_output_path = BANDS_OUTPUT_DIR / f"{tile_name}_{band_name}.npy"
        
        # Save as numpy array (preserves float32 precision)
        np.save(band_output_path, tile_data.astype(np.float32))

print(f"Multispectral bands saved for {len(valid_tile_indices)} tiles")

## Process Vegetation Indices

In [ ]:
print("\nProcessing vegetation indices...")

# Get all index files
index_files = {
    'ndvi': INDEX_MAP_DIR / 'NDVI.tif',
    'gndvi': INDEX_MAP_DIR / 'GNDVI.tif',
    'ndre': INDEX_MAP_DIR / 'NDRE.tif',
    'lci': INDEX_MAP_DIR / 'LCI.tif',
    'osavi': INDEX_MAP_DIR / 'OSAVI.tif'
}

for index_name, index_path in index_files.items():
    if not index_path.exists():
        print(f"Warning: {index_path} not found, skipping...")
        continue
    
    print(f"Processing {index_name.upper()} index...")
    index_tiles = extract_tiles_from_raster(index_path, TILE_SIZE, OVERLAP)
    
    for tile_data, row_idx, col_idx in tqdm(index_tiles, desc=f"Tiling {index_name.upper()}"):
        # Only save tiles that passed RGB quality check
        if (row_idx, col_idx) not in valid_tile_indices:
            continue
        
        tile_name = f"tile_{row_idx:04d}_{col_idx:04d}"
        index_output_path = INDEX_OUTPUT_DIR / f"{tile_name}_{index_name}.npy"
        
        # Save as numpy array
        np.save(index_output_path, tile_data.astype(np.float32))

print(f"Vegetation indices saved for {len(valid_tile_indices)} tiles")

## Save Metadata

In [ ]:
import json

# Save tile metadata
metadata_path = METADATA_OUTPUT_DIR / 'tile_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump({
        'flight_name': FLIGHT_NAME,
        'tile_size': TILE_SIZE,
        'overlap': OVERLAP,
        'total_tiles': len(valid_tile_indices),
        'min_brightness': MIN_BRIGHTNESS,
        'min_std': MIN_STD,
        'bands': list(band_mapping.keys()),
        'indices': list(index_files.keys()),
        'tiles': tile_metadata
    }, f, indent=2)

print(f"\nMetadata saved to {metadata_path}")

## Visualization: Sample Tiles

In [ ]:
# Visualize a few sample tiles
import random

sample_tiles = random.sample(tile_metadata, min(6, len(tile_metadata)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, tile_info in enumerate(sample_tiles):
    tile_name = tile_info['tile_name']
    rgb_path = RGB_OUTPUT_DIR / f"{tile_name}.png"
    
    img = cv2.imread(str(rgb_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    axes[idx].imshow(img_rgb)
    axes[idx].set_title(f"{tile_name}\nBrightness: {tile_info['mean_brightness']:.1f}")
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig(METADATA_OUTPUT_DIR / 'sample_tiles.png', dpi=150, bbox_inches='tight')
plt.show()

print("Sample tiles visualization saved")

## Visualization: Index Comparison

In [ ]:
# Visualize one tile with all its indices
sample_tile = tile_metadata[len(tile_metadata)//2]  # Pick middle tile
tile_name = sample_tile['tile_name']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# RGB
rgb_path = RGB_OUTPUT_DIR / f"{tile_name}.png"
img = cv2.imread(str(rgb_path))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
axes[0, 0].imshow(img_rgb)
axes[0, 0].set_title('RGB', fontsize=14, fontweight='bold')
axes[0, 0].axis('off')

# NIR
nir_path = BANDS_OUTPUT_DIR / f"{tile_name}_nir.npy"
if nir_path.exists():
    nir_data = np.load(nir_path)
    axes[0, 1].imshow(nir_data, cmap='gray')
    axes[0, 1].set_title('NIR Band', fontsize=14, fontweight='bold')
    axes[0, 1].axis('off')

# RedEdge
rededge_path = BANDS_OUTPUT_DIR / f"{tile_name}_rededge.npy"
if rededge_path.exists():
    rededge_data = np.load(rededge_path)
    axes[0, 2].imshow(rededge_data, cmap='gray')
    axes[0, 2].set_title('RedEdge Band', fontsize=14, fontweight='bold')
    axes[0, 2].axis('off')

# Green
green_path = BANDS_OUTPUT_DIR / f"{tile_name}_green.npy"
if green_path.exists():
    green_data = np.load(green_path)
    axes[0, 3].imshow(green_data, cmap='gray')
    axes[0, 3].set_title('Green Band', fontsize=14, fontweight='bold')
    axes[0, 3].axis('off')

# Indices
index_names = ['ndvi', 'gndvi', 'ndre', 'lci']
for idx, index_name in enumerate(index_names):
    index_path = INDEX_OUTPUT_DIR / f"{tile_name}_{index_name}.npy"
    if index_path.exists():
        index_data = np.load(index_path)
        im = axes[1, idx].imshow(index_data, cmap='RdYlGn', vmin=-1, vmax=1)
        axes[1, idx].set_title(index_name.upper(), fontsize=14, fontweight='bold')
        axes[1, idx].axis('off')
        plt.colorbar(im, ax=axes[1, idx], fraction=0.046, pad=0.04)

plt.suptitle(f'Sample Tile: {tile_name}', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(METADATA_OUTPUT_DIR / 'index_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("Index comparison visualization saved")

## Summary Statistics

In [ ]:
print("\n" + "="*60)
print("PREPROCESSING SUMMARY")
print("="*60)
print(f"Flight: {FLIGHT_NAME}")
print(f"Tile size: {TILE_SIZE}x{TILE_SIZE}")
print(f"Overlap: {OVERLAP} pixels")
print(f"\nTotal valid tiles: {len(valid_tile_indices)}")
print(f"\nOutput directories:")
print(f"  RGB tiles (for CVAT): {RGB_OUTPUT_DIR}")
print(f"  Multispectral bands: {BANDS_OUTPUT_DIR}")
print(f"  Vegetation indices: {INDEX_OUTPUT_DIR}")
print(f"  Metadata: {METADATA_OUTPUT_DIR}")
print("\nAvailable bands:")
for band_name in band_mapping.keys():
    print(f"  - {band_name}")
print("\nAvailable indices:")
for index_name in index_files.keys():
    print(f"  - {index_name.upper()}")
print("\n" + "="*60)
print("NEXT STEPS:")
print("1. Upload RGB tiles from 'rgb_tiles/' to CVAT for labeling")
print("2. Label vegetation and power lines in CVAT")
print("3. Export masks from CVAT")
print("4. Use corresponding .npy files for training with multispectral data")
print("="*60)